# Assignment 1b: Image Filtering with Convolutions

  saves filtered images under `output/part1b_filtering/`.


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

image_path = Path('assignment-1-data/goldfish.jpg')
if not image_path.is_file():
    raise FileNotFoundError(f'Could not find {image_path}. Run this notebook from the assignment folder.')

with Image.open(image_path) as source:
    image = np.asarray(source.convert('RGB'))


## Part 1b: Image Filtering with Convolutions

Create normalized square box filters, flatten each local image window and the filter before taking dot products, apply the filter to each color channel independently, and compare multiple filter sizes. Reflection padding preserves the image dimensions.


In [ ]:
def make_box_filter(s):
    """Return a normalized s-by-s single-channel box filter."""
    if not isinstance(s, (int, np.integer)):
        raise TypeError('s must be an integer')
    if s <= 0:
        raise ValueError('s must be a positive integer')
    return np.full((s, s), 1.0 / (s * s), dtype=np.float64)


def convolve_image(image, filter):
    """Apply a square 2D filter independently to each image channel."""
    image = np.asarray(image)
    kernel = np.asarray(filter, dtype=np.float64)
    if image.ndim not in (2, 3):
        raise ValueError('image must be grayscale (2D) or channel-last (3D)')
    if image.shape[0] == 0 or image.shape[1] == 0:
        raise ValueError('image dimensions must be nonzero')
    if kernel.ndim != 2 or kernel.shape[0] != kernel.shape[1]:
        raise ValueError('filter must be a square 2D array')
    if kernel.shape[0] == 0:
        raise ValueError('filter size must be positive')
    if image.ndim == 3 and image.shape[2] == 0:
        raise ValueError('image must contain at least one channel')

    height, width = image.shape[:2]
    filter_size = kernel.shape[0]
    top = left = filter_size // 2
    bottom = right = filter_size - 1 - filter_size // 2
    padding = ((top, bottom), (left, right))
    if image.ndim == 3:
        padding += ((0, 0),)

    pad_mode = 'reflect' if height > 1 and width > 1 else 'edge'
    padded = np.pad(image, padding, mode=pad_mode)
    result = np.empty(image.shape, dtype=np.float64)
    flat_filter = kernel.flatten()

    for y in range(height):
        windows = np.lib.stride_tricks.sliding_window_view(
            padded[y:y + filter_size, ...],
            (filter_size, filter_size),
            axis=(0, 1),
        )[0]
        if image.ndim == 2:
            flat_windows = np.stack([window.flatten() for window in windows])
        else:
            channels = image.shape[2]
            flat_windows = np.empty((width, channels, filter_size * filter_size))
            for channel in range(channels):
                flat_windows[:, channel, :] = np.stack(
                    [window.flatten() for window in windows[:, channel]]
                )
        result[y] = np.dot(flat_windows, flat_filter)

    if np.issubdtype(image.dtype, np.integer):
        limits = np.iinfo(image.dtype)
        result = np.rint(result).clip(limits.min, limits.max)
    return result.astype(image.dtype, copy=False)


In [ ]:
filter_sizes = (3, 7, 15)
filter_output_dir = Path('output/part1b_filtering')
filter_output_dir.mkdir(parents=True, exist_ok=True)

filtered_images = {}
for size in filter_sizes:
    box_filter = make_box_filter(size)
    filtered_images[size] = convolve_image(image, box_filter)
    output_path = filter_output_dir / f'goldfish_box_blur_{size}x{size}.png'
    Image.fromarray(filtered_images[size]).save(output_path)
    print(f'{size}x{size} filter sum: {box_filter.sum():.1f}; saved {output_path}')


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(image)
axes[0].set_title('Original')
axes[0].axis('off')
for axis, size in zip(axes[1:], filter_sizes):
    axis.imshow(filtered_images[size])
    axis.set_title(f'{size}x{size} box filter')
    axis.axis('off')
plt.tight_layout()
plt.show()


### Part 1b observations

- The 3x3 filter gives a mild blur while keeping most larger features clear.
- The 7x7 filter smooths the image more noticeably.
- The 15x15 filter strongly softens fine detail and boundaries. Larger averaging windows remove more high-frequency detail.
